# Backtesting Results Visualization

**Purpose**: Evaluate model performance using walk-forward validation

**Methodology**:
- Walk-forward validation with rolling windows
- 30-day training window, 5-day test window
- 5-day step size
- No data leakage (verified)

**Metrics Tracked**:
- MAE, RMSE, MAPE per fold
- Directional accuracy per fold
- Cumulative performance
- Model stability over time

**Expected Runtime**: ~10-20 minutes (model retraining on each fold)

**Prerequisites**:
- Trained model(s) from model_comparison.ipynb
- 60+ days of data for multiple folds

In [6]:
"""Import libraries and configure backtesting parameters."""
from __future__ import annotations

import asyncio
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from loguru import logger
from plotly.subplots import make_subplots
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
from rich.table import Table

from price_stradamus.config.constants import (
    FEATURE_WARMUP_PERIOD,
    REALISTIC_DIR_ACC_MAX,
    SUSPICIOUS_DIR_ACC,
    SUSPICIOUS_SHARPE,
)
from price_stradamus.config.settings import settings
from price_stradamus.data.database import DatabaseManager
from price_stradamus.data.preprocessor import DataPreprocessor

# Use StatefulFeatureEngineer - NOT the deprecated FeatureEngineer!
from price_stradamus.data.stateful_features import StatefulFeatureEngineer
from price_stradamus.evaluation.metrics import MetricsCalculator
from price_stradamus.evaluation.financial_metrics import TradingCosts
from price_stradamus.models.registry import ModelRegistry
from price_stradamus.visualization.charts import PriceChartVisualizer
from price_stradamus.utils.feature_selection import CORE_FEATURES  # NEW: Import core features

console = Console()

# Backtesting configuration
BACKTEST_CONFIG = {
    "model_name": "xgboost",  # Change to your best model
    "symbol": "BTCUSDT",
    "timeframe": "1m",
    "days": 120,  # Fetch 4 months of data
    "train_window_days": 90,  # 3 months
    "test_window_hours": 1,  # 3 hours
    "max_folds": 3,  # Limit number of folds for faster backtesting
    "input_chunk_length": 120, # 60 default baseline
    "output_chunk_length": 10, # 5 default baseline
    "n_epochs": 30,
}

# Trading costs for realistic evaluation
COSTS = TradingCosts(commission_pct=0.001, slippage_pct=0.0005)

console.print("[bold cyan]Backtesting Configuration[/bold cyan]")
console.print(f"Model: {BACKTEST_CONFIG['model_name']}")
console.print(
    f"Walk-forward: {BACKTEST_CONFIG['train_window_days']}-day train, {BACKTEST_CONFIG['test_window_hours']}-hour test"
)
console.print(f"Max folds: {BACKTEST_CONFIG['max_folds']}")
console.print(f"Transaction costs: {COSTS.round_trip_cost * 100:.2f}% per round trip")

Backtesting Configuration

Model: xgboost

Walk-forward: 90-day train, 1-hour test

Max folds: 3

Transaction costs: 0.30% per round trip

In [7]:
"""Load extended historical data for walk-forward validation.

IMPORTANT: DO NOT generate features here!
Features are generated SEPARATELY for each fold.
"""

console.print("[cyan]Loading historical data for backtesting...[/cyan]")

db = DatabaseManager(settings.database_url)
await db.initialize()

end_date = datetime.now()
start_date = end_date - timedelta(days=BACKTEST_CONFIG["days"])

df_raw = await db.get_ohlcv(
    symbol=BACKTEST_CONFIG["symbol"],
    timeframe=BACKTEST_CONFIG["timeframe"],
    start_date=start_date,
    end_date=end_date,
)

await db.close()

console.print(f"[green]✓ Loaded {len(df_raw)} candles[/green]")
console.print(
    f"   Date range: {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}"
)

# Preprocess (validation, missing values only - NO feature generation!)
preprocessor = DataPreprocessor()
df_clean = preprocessor.validate_ohlcv(df_raw)
df_clean = preprocessor.handle_missing_values(df_clean)

# Ensure timestamp is datetime
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], utc=True)

console.print(f"[green]✓ Preprocessed {len(df_clean)} rows[/green]")
console.print(
    f"[yellow]⚠️  Features will be generated SEPARATELY for each fold[/yellow]"
)

Loading historical data for backtesting...

2025-12-15 16:35:32.835 | INFO     | price_stradamus.data.database:__init__:57 - DatabaseManager initialized
2025-12-15 16:35:32.838 | INFO     | price_stradamus.data.database:initialize:82 - Database engine and session maker initialized (pool_size=10, max_overflow=20)
2025-12-15 16:35:32.840 | DEBUG    | price_stradamus.data.database:get_ohlcv:196 - Querying OHLCV data for BTCUSDT 1m
2025-12-15 16:35:42.370 | INFO     | price_stradamus.data.database:get_ohlcv:251 - Retrieved 161530 OHLCV records for BTCUSDT 1m
2025-12-15 16:35:42.473 | INFO     | price_stradamus.data.database:close:92 - Database connections closed


✓ Loaded 161530 candles

Date range: 2025-08-17 09:36:00+00:00 to 2025-12-07 13:45:00+00:00

2025-12-15 16:35:42.488 | INFO     | price_stradamus.data.preprocessor:__init__:60 - DataPreprocessor initialized (normalization=minmax)
2025-12-15 16:35:42.589 | DEBUG    | price_stradamus.data.preprocessor:validate_ohlcv:138 - OHLCV validation passed for 161530 rows
2025-12-15 16:35:42.600 | DEBUG    | price_stradamus.data.preprocessor:handle_missing_values:167 - No missing values found


✓ Preprocessed 161530 rows

⚠️  Features will be generated SEPARATELY for each fold

In [8]:
"""Create walk-forward windows and preview folds."""

# Calculate samples per day/hour
SAMPLES_PER_DAY = 24 * 60  # 1440 for 1-minute data
SAMPLES_PER_HOUR = 60  # 60 for 1-minute data

train_samples = BACKTEST_CONFIG["train_window_days"] * SAMPLES_PER_DAY
test_samples = BACKTEST_CONFIG["test_window_hours"] * SAMPLES_PER_HOUR

# Calculate number of possible folds
total_data = len(df_clean)
available_for_testing = total_data - train_samples
possible_folds = max(1, available_for_testing // test_samples)

# Limit to max_folds by selecting evenly spaced folds
max_folds = BACKTEST_CONFIG.get("max_folds", possible_folds)
num_folds = min(max_folds, possible_folds)

# Create evenly spaced fold indices
if num_folds < possible_folds:
    # Select evenly distributed folds across the entire range
    fold_indices = np.linspace(0, possible_folds - 1, num_folds, dtype=int)
    console.print(
        f"\n[yellow]Note: Selecting {num_folds} evenly spaced folds from {possible_folds} possible folds[/yellow]"
    )
else:
    fold_indices = list(range(num_folds))

console.print(
    f"\n[bold]Walk-Forward Folds: {num_folds} (of {possible_folds} possible)[/bold]"
)
console.print(
    f"  Train window: {BACKTEST_CONFIG['train_window_days']} days ({train_samples} samples)"
)
console.print(
    f"  Test window: {BACKTEST_CONFIG['test_window_hours']} hours ({test_samples} samples)"
)

# Display first 3 folds
table = Table(title=f"Walk-Forward Windows (First 3 of {num_folds})")
table.add_column("Fold #", style="cyan")
table.add_column("Train Start", style="green")
table.add_column("Train End", style="green")
table.add_column("Test Start", style="yellow")
table.add_column("Test End", style="yellow")

for idx, fold_idx in enumerate(fold_indices[:3]):
    test_end_idx = total_data - (possible_folds - fold_idx - 1) * test_samples
    test_start_idx = test_end_idx - test_samples
    train_end_idx = test_start_idx
    train_start_idx = max(0, train_end_idx - train_samples)

    train_start = df_clean.iloc[train_start_idx]["timestamp"]
    train_end = df_clean.iloc[train_end_idx - 1]["timestamp"]
    test_start = df_clean.iloc[test_start_idx]["timestamp"]
    test_end = df_clean.iloc[min(test_end_idx - 1, len(df_clean) - 1)]["timestamp"]

    table.add_row(
        str(idx + 1),
        str(train_start)[:16],
        str(train_end)[:16],
        str(test_start)[:16],
        str(test_end)[:16],
    )

console.print(table)

if num_folds > 3:
    console.print(f"[dim]... and {num_folds - 3} more folds[/dim]")

Note: Selecting 3 evenly spaced folds from 532 possible folds

Walk-Forward Folds: 3 (of 532 possible)

Train window: 90 days (129600 samples)

Test window: 1 hours (60 samples)

                         Walk-Forward Windows (First 3 of 3)                          
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Fold # ┃ Train Start      ┃ Train End        ┃ Test Start       ┃ Test End         ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 1      │ 2025-08-17 09:46 │ 2025-11-15 09:45 │ 2025-11-15 09:46 │ 2025-11-15 10:45 │
│ 2      │ 2025-08-28 10:46 │ 2025-11-26 10:45 │ 2025-11-26 10:46 │ 2025-11-26 11:45 │
│ 3      │ 2025-09-08 12:46 │ 2025-12-07 12:45 │ 2025-12-07 12:46 │ 2025-12-07 13:45 │
└────────┴──────────────────┴──────────────────┴──────────────────┴──────────────────┘

In [9]:
"""Execute walk-forward validation with proper feature engineering per fold."""

backtest_results = []

with Progress(
    SpinnerColumn(),
    TextColumn("[progress.description]{task.description}"),
    BarColumn(),
    TextColumn("[progress.percentage]{task.percentage:>3.0f}%"),
    console=console,
) as progress:
    task = progress.add_task(
        "[cyan]Running walk-forward backtesting...",
        total=num_folds,
    )

    for idx, fold_idx in enumerate(fold_indices):
        
        progress.update(task, description=f"[cyan]Fold {idx + 1}/{num_folds}...")

        try:
            # Calculate fold boundaries using the selected fold index
            test_end_idx = total_data - (possible_folds - fold_idx - 1) * test_samples
            test_start_idx = test_end_idx - test_samples
            train_end_idx = test_start_idx
            train_start_idx = max(0, train_end_idx - train_samples)

            # Split RAW data for this fold
            fold_train_df = df_clean.iloc[train_start_idx:train_end_idx].copy()
            fold_test_df = df_clean.iloc[test_start_idx:test_end_idx].copy()

            # CRITICAL: Create NEW StatefulFeatureEngineer for each fold
            fold_engineer = StatefulFeatureEngineer(warmup_period=FEATURE_WARMUP_PERIOD, feature_cols=CORE_FEATURES)

            # fit_transform on this fold's training data
            fold_train_features = fold_engineer.fit_transform(fold_train_df)
            # transform test using this fold's training statistics
            fold_test_features = fold_engineer.transform(
                fold_test_df, drop_warmup=False
            )

            # DEBUG: Check what features exist (only for first fold)
            if idx == 0:
                console.print(f"\n[cyan]DEBUG - Fold 1 feature check:[/cyan]")
                console.print(f"  Train features shape: {fold_train_features.shape}")
                console.print(f"  Available columns: {list(fold_train_features.columns)}... (showing {len(fold_train_features.columns)})")

                # Check which CORE_FEATURES are missing
                missing_features = [f for f in CORE_FEATURES if f not in fold_train_features.columns]
                available_features = [f for f in CORE_FEATURES if f in fold_train_features.columns]

                console.print(f"  CORE_FEATURES available: {len(available_features)}/{len(CORE_FEATURES)}")
                if missing_features:
                    console.print(f"  [yellow]Missing features: {missing_features}[/yellow]")

                # Check for NaN values in available features
                if available_features:
                    nan_counts = fold_train_features[available_features].isna().sum()
                    features_with_nans = nan_counts[nan_counts > 0]
                    if len(features_with_nans) > 0:
                        console.print(f"  [yellow]Features with NaN values:[/yellow]")
                        for feat, count in features_with_nans.items():
                            console.print(f"    - {feat}: {count} NaN values ({count/len(fold_train_features)*100:.1f}%)")
            else:
                break
            # # Convert to time series with returns target and covariates
            # train_ts, train_cov = fold_engineer.to_darts_timeseries(
            #     fold_train_features,
            #     value_cols=["returns_1"],  # Predict returns instead of prices
            #     covariates=CORE_FEATURES,  # Use 18 core features as covariates
            # )
            # test_ts, test_cov = fold_engineer.to_darts_timeseries(
            #     fold_test_features,
            #     value_cols=["returns_1"],
            #     covariates=CORE_FEATURES,
            # )

            # # Validate covariates were created
            # if train_cov is None or test_cov is None:
            #     console.print(f"[yellow]⚠ Fold {idx + 1}: Covariates not available, skipping[/yellow]")
            #     logger.warning(f"Fold {idx + 1}: Covariates could not be created")
            #     continue

            # # Ensure both train and test covariates have the same features
            # train_cov_cols = set(train_cov.components)
            # test_cov_cols = set(test_cov.components)
            # common_cov_cols = sorted(train_cov_cols & test_cov_cols)  # Intersection

            # if len(common_cov_cols) == 0:
            #     console.print(f"[yellow]⚠ Fold {idx + 1}: No common covariates, skipping[/yellow]")
            #     logger.warning(f"Fold {idx + 1}: No common covariates between train and test")
            #     continue

            # # Filter both to common features only
            # if len(common_cov_cols) < len(train_cov.components):
            #     dropped_features = sorted(train_cov_cols - test_cov_cols)
            #     logger.info(f"Fold {idx + 1}: Using {len(common_cov_cols)} common features (dropped: {dropped_features})")
            #     train_cov = train_cov[common_cov_cols]
            #     test_cov = test_cov[common_cov_cols]

            # # Store last known price for inverse transformation
            # last_train_price = fold_train_features["close"].iloc[-1]

            # # Create and train model
            # model = ModelRegistry.create_model(
            #     BACKTEST_CONFIG["model_name"],
            #     input_chunk_length=BACKTEST_CONFIG["input_chunk_length"],
            #     output_chunk_length=BACKTEST_CONFIG["output_chunk_length"],
            #     n_epochs=BACKTEST_CONFIG["n_epochs"],
            # )

            # # Train with covariates and last known price
            # model.fit(
            #     train_ts,
            #     past_covariates=train_cov,
            #     last_known_price=last_train_price,
            # )

            # # Predict with covariates (concatenate train + test covariates)
            # all_covariates = train_cov.append(test_cov)
            # predictions = model.predict(
            #     n=len(test_ts),
            #     series=train_ts,
            #     past_covariates=all_covariates,
            # )

            # # Extract actual prices for comparison
            # # (Predictions are already in price space after inverse transform)
            # actuals = fold_test_features["close"].values[:len(test_ts)]
            # preds = predictions.values().flatten()[: len(actuals)]

            # fold_metrics = MetricsCalculator.calculate_all_with_financial(
            #     actuals, preds, include_financial=True, trading_costs=COSTS
            # )

            # # Store results
            # backtest_results.append(
            #     {
            #         "fold": idx + 1,
            #         "train_start": fold_train_df["timestamp"].min(),
            #         "train_end": fold_train_df["timestamp"].max(),
            #         "test_start": fold_test_df["timestamp"].min(),
            #         "test_end": fold_test_df["timestamp"].max(),
            #         "metrics": fold_metrics,
            #         "predictions": preds,
            #         "actuals": actuals,
            #         "timestamps": test_ts.time_index,
            #     }
            # )

        except Exception as e:
            console.print(f"[red]✗ Fold {idx + 1} failed: {e}[/red]")
            logger.error(f"Fold {idx + 1} error: {e}")
            import traceback
            logger.error(traceback.format_exc())

        progress.advance(task)

console.print(
    f"\n[green]✓ Completed {len(backtest_results)} folds with proper feature regeneration[/green]"
)
console.print(
    f"[green]✓ Each fold used StatefulFeatureEngineer - NO DATA LEAKAGE![/green]"
)

Output()

2025-12-15 16:35:42.872 | INFO     | price_stradamus.data.stateful_features:__init__:151 - StatefulFeatureEngineer initialized (warmup_period=200, feature_cols=24 selected)
2025-12-15 16:35:42.876 | INFO     | price_stradamus.data.stateful_features:fit:175 - Fitting feature statistics on 129600 training samples


2025-12-15 16:35:48.732 | INFO     | price_stradamus.data.stateful_features:_filter_features:661 - Feature filtering: keeping 24/27 features (+ 8 base columns)


2025-12-15 16:35:49.444 | INFO     | price_stradamus.data.stateful_features:fit:217 - Fitted statistics for 26 features
2025-12-15 16:35:49.453 | DEBUG    | price_stradamus.data.stateful_features:transform:249 - Transforming 129600 samples


2025-12-15 16:35:53.592 | INFO     | price_stradamus.data.stateful_features:_filter_features:661 - Feature filtering: keeping 24/27 features (+ 8 base columns)
2025-12-15 16:35:53.658 | DEBUG    | price_stradamus.data.stateful_features:transform:249 - Transforming 60 samples


...

2025-12-15 16:35:56.885 | INFO     | price_stradamus.data.stateful_features:fit:217 - Fitted statistics for 26 features
2025-12-15 16:35:56.888 | DEBUG    | price_stradamus.data.stateful_features:transform:249 - Transforming 129600 samples


2025-12-15 16:35:59.703 | INFO     | price_stradamus.data.stateful_features:_filter_features:661 - Feature filtering: keeping 24/27 features (+ 8 base columns)
2025-12-15 16:35:59.775 | DEBUG    | price_stradamus.data.stateful_features:transform:249 - Transforming 60 samples


2025-12-15 16:35:59.850 | INFO     | price_stradamus.data.stateful_features:_filter_features:661 - Feature filtering: keeping 24/27 features (+ 8 base columns)


✓ Completed 0 folds with proper feature regeneration

✓ Each fold used StatefulFeatureEngineer - NO DATA LEAKAGE!

In [10]:
"""Calculate aggregate statistics across all folds including financial metrics."""

# Extract metrics from all folds
mae_values = [r["metrics"]["mae"] for r in backtest_results]
rmse_values = [r["metrics"]["rmse"] for r in backtest_results]
mape_values = [r["metrics"]["mape"] for r in backtest_results]
dir_acc_values = [r["metrics"]["directional_accuracy"] for r in backtest_results]
sharpe_values = [r["metrics"]["fin_sharpe_ratio"] for r in backtest_results]
pnl_values = [r["metrics"]["fin_net_pnl"] for r in backtest_results]
win_rate_values = [r["metrics"]["fin_win_rate"] for r in backtest_results]

# Calculate statistics
stats = {
    "MAE": {
        "mean": np.mean(mae_values),
        "std": np.std(mae_values),
        "min": np.min(mae_values),
        "max": np.max(mae_values),
    },
    "RMSE": {
        "mean": np.mean(rmse_values),
        "std": np.std(rmse_values),
        "min": np.min(rmse_values),
        "max": np.max(rmse_values),
    },
    "MAPE": {
        "mean": np.mean(mape_values),
        "std": np.std(mape_values),
        "min": np.min(mape_values),
        "max": np.max(mape_values),
    },
    "Dir Acc": {
        "mean": np.mean(dir_acc_values),
        "std": np.std(dir_acc_values),
        "min": np.min(dir_acc_values),
        "max": np.max(dir_acc_values),
    },
    "Sharpe": {
        "mean": np.mean(sharpe_values),
        "std": np.std(sharpe_values),
        "min": np.min(sharpe_values),
        "max": np.max(sharpe_values),
    },
    "Net P&L": {
        "mean": np.mean(pnl_values),
        "std": np.std(pnl_values),
        "min": np.min(pnl_values),
        "max": np.max(pnl_values),
    },
    "Win Rate": {
        "mean": np.mean(win_rate_values),
        "std": np.std(win_rate_values),
        "min": np.min(win_rate_values),
        "max": np.max(win_rate_values),
    },
}

# Display results table
table = Table(
    title=f"Walk-Forward Backtest Results (({BACKTEST_CONFIG['model_name']}-{BACKTEST_CONFIG['max_folds']} F-{BACKTEST_CONFIG['input_chunk_length']} ICL-{BACKTEST_CONFIG['output_chunk_length']} OCL-{BACKTEST_CONFIG['n_epochs']} n_E), Costs: {COSTS.round_trip_cost * 100:.2f}%)"
)

table.add_column("Metric", style="cyan")
table.add_column("Mean", style="green")
table.add_column("Std Dev", style="yellow")
table.add_column("Min", style="magenta")
table.add_column("Max", style="magenta")

for metric, values in stats.items():
    if metric == "Net P&L":
        table.add_row(
            metric,
            f"${values['mean']:.2f}",
            f"±${values['std']:.2f}",
            f"${values['min']:.2f}",
            f"${values['max']:.2f}",
        )
    elif metric in ["Dir Acc", "Win Rate", "MAPE"]:
        table.add_row(
            metric,
            f"{values['mean']:.2f}%",
            f"±{values['std']:.2f}%",
            f"{values['min']:.2f}%",
            f"{values['max']:.2f}%",
        )
    else:
        table.add_row(
            metric,
            f"{values['mean']:.2f}",
            f"±{values['std']:.2f}",
            f"{values['min']:.2f}",
            f"{values['max']:.2f}",
        )

console.print(table)

console.print(f"\n[bold]Key Observations:[/bold]")
console.print(f"  - Evaluated on {len(backtest_results)} independent folds")
console.print(
    f"  - Average RMSE: {stats['RMSE']['mean']:.2f} ±{stats['RMSE']['std']:.2f} USDT"
)
console.print(f"  - Average Directional Accuracy: {stats['Dir Acc']['mean']:.2f}%")
console.print(f"  - Average Sharpe Ratio: {stats['Sharpe']['mean']:.2f}")
console.print(f"  - Average Net P&L: ${stats['Net P&L']['mean']:.2f}")

# Model stability
rmse_cv = (
    (stats["RMSE"]["std"] / stats["RMSE"]["mean"]) * 100
    if stats["RMSE"]["mean"] > 0
    else 0
)
console.print(f"  - Model stability (RMSE CV): {rmse_cv:.1f}%")

d:\priceStradamus\price-stradamus\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\priceStradamus\price-stradamus\.venv\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
d:\priceStradamus\price-stradamus\.venv\Lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
d:\priceStradamus\price-stradamus\.venv\Lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
d:\priceStradamus\price-stradamus\.venv\Lib\site-packages\numpy\_core\_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: zero-size array to reduction operation minimum which has no identity

In [ ]:
"""Plot how metrics change across folds (time)."""

fold_numbers = [r["fold"] for r in backtest_results]
test_end_dates = [r["test_end"] for r in backtest_results]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "MAE Over Time",
        "RMSE Over Time",
        "MAPE Over Time",
        "Directional Accuracy Over Time",
    ),
)

# MAE
fig.add_trace(
    go.Scatter(
        x=test_end_dates,
        y=mae_values,
        mode="lines+markers",
        name="MAE",
        line=dict(color="cyan", width=2),
        marker=dict(size=8),
    ),
    row=1,
    col=1,
)

# RMSE
fig.add_trace(
    go.Scatter(
        x=test_end_dates,
        y=rmse_values,
        mode="lines+markers",
        name="RMSE",
        line=dict(color="orange", width=2),
        marker=dict(size=8),
    ),
    row=1,
    col=2,
)

# MAPE
fig.add_trace(
    go.Scatter(
        x=test_end_dates,
        y=mape_values,
        mode="lines+markers",
        name="MAPE",
        line=dict(color="yellow", width=2),
        marker=dict(size=8),
    ),
    row=2,
    col=1,
)

# Directional Accuracy
fig.add_trace(
    go.Scatter(
        x=test_end_dates,
        y=dir_acc_values,
        mode="lines+markers",
        name="Dir Acc",
        line=dict(color="green", width=2),
        marker=dict(size=8),
    ),
    row=2,
    col=2,
)

# Add mean lines
fig.add_hline(
    y=stats["MAE"]["mean"],
    line_dash="dash",
    line_color="cyan",
    opacity=0.5,
    row=1,
    col=1,
)
fig.add_hline(
    y=stats["RMSE"]["mean"],
    line_dash="dash",
    line_color="orange",
    opacity=0.5,
    row=1,
    col=2,
)
fig.add_hline(
    y=stats["MAPE"]["mean"],
    line_dash="dash",
    line_color="yellow",
    opacity=0.5,
    row=2,
    col=1,
)
fig.add_hline(
    y=stats["Dir Acc"]["mean"],
    line_dash="dash",
    line_color="green",
    opacity=0.5,
    row=2,
    col=2,
)

fig.update_layout(
    title_text=f"Backtesting Performance Over Time - {BACKTEST_CONFIG['model_name']}",
    showlegend=False,
    height=800,
    template="plotly_dark",
)

fig.update_xaxes(title_text="Test Period End Date")
fig.show()

In [ ]:
"""Display predictions for best and worst performing folds."""

# Find best and worst folds by RMSE
best_fold = min(backtest_results, key=lambda x: x["metrics"]["rmse"])
worst_fold = max(backtest_results, key=lambda x: x["metrics"]["rmse"])

console.print(
    f"[bold green]Best Fold: #{best_fold['fold']}[/bold green] (RMSE: {best_fold['metrics']['rmse']:.2f})"
)
console.print(
    f"[bold red]Worst Fold: #{worst_fold['fold']}[/bold red] (RMSE: {worst_fold['metrics']['rmse']:.2f})"
)

# Plot both folds
for fold_result, title_suffix, color in [
    (best_fold, "Best", "green"),
    (worst_fold, "Worst", "red"),
]:
    fig = go.Figure()

    # Actual prices
    fig.add_trace(
        go.Scatter(
            x=fold_result["timestamps"],
            y=fold_result["actuals"],
            mode="lines+markers",
            name="Actual",
            line=dict(color="white", width=3),
            marker=dict(size=6),
        )
    )

    # Predicted prices
    fig.add_trace(
        go.Scatter(
            x=fold_result["timestamps"],
            y=fold_result["predictions"],
            mode="lines+markers",
            name="Predicted",
            line=dict(color=color, width=2, dash="dash"),
            marker=dict(size=6),
        )
    )

    fig.update_layout(
        title=f"Fold #{fold_result['fold']} - {title_suffix} (RMSE: {fold_result['metrics']['rmse']:.2f} USDT) (MAE: {fold_result['metrics']['mae']:.2f} USDT) ({BACKTEST_CONFIG['model_name']}-{BACKTEST_CONFIG['max_folds']} F-{BACKTEST_CONFIG['input_chunk_length']} ICL-{BACKTEST_CONFIG['output_chunk_length']} OCL-{BACKTEST_CONFIG['n_epochs']} n_E)",
        xaxis_title="Time",
        yaxis_title="Price (USDT)",
        template="plotly_dark",
        height=500,
        hovermode="x unified",
    )

    fig.show()

In [ ]:
"""Analyze cumulative prediction errors across all folds."""

# Concatenate all predictions and actuals
all_actuals = np.concatenate([r["actuals"] for r in backtest_results])
all_predictions = np.concatenate([r["predictions"] for r in backtest_results])
all_timestamps = pd.DatetimeIndex(
    np.concatenate([r["timestamps"] for r in backtest_results])
)

# Calculate cumulative error
cumulative_error = np.cumsum(all_predictions - all_actuals)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=all_timestamps,
        y=cumulative_error,
        mode="lines",
        name="Cumulative Error",
        line=dict(color="red", width=2),
        fill="tozeroy",
        fillcolor="rgba(255, 0, 0, 0.2)",
    )
)

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.5)

fig.update_layout(
    title="Cumulative Prediction Error Over Time",
    xaxis_title="Time",
    yaxis_title="Cumulative Error (USDT)",
    template="plotly_dark",
    height=600,
)

fig.show()

# Calculate bias
mean_error = np.mean(all_predictions - all_actuals)
console.print(f"\n[bold]Model Bias Analysis:[/bold]")
console.print(f"  Mean error: {mean_error:+.2f} USDT")
if abs(mean_error) < 1:
    console.print(f"  [green]✓ Low bias - model is well-calibrated[/green]")
else:
    bias_direction = "over-predicting" if mean_error > 0 else "under-predicting"
    console.print(
        f"  [yellow]⚠ Model is {bias_direction} by {abs(mean_error):.2f} USDT on average[/yellow]"
    )

## Backtesting Summary & Production Readiness

Results from walk-forward validation provide unbiased performance estimates.

In [ ]:
"""Final recommendations based on backtest results."""

console.print("=" * 70)
console.print(" BACKTESTING RESULTS - PRODUCTION READINESS ")
console.print("=" * 70)

# Overall assessment
console.print(
    f"\n1. MODEL PERFORMANCE: [bold]{BACKTEST_CONFIG['model_name'].upper()}[/bold]"
)
console.print(
    f"   - Average RMSE: {stats['RMSE']['mean']:.2f} ±{stats['RMSE']['std']:.2f} USDT"
)
console.print(f"   - Average MAE: {stats['MAE']['mean']:.2f} USDT")
console.print(f"   - Average Directional Accuracy: {stats['Dir Acc']['mean']:.2f}%")
console.print(f"   - Evaluated on: {len(backtest_results)} independent folds")

# Stability assessment
rmse_cv = (stats["RMSE"]["std"] / stats["RMSE"]["mean"]) * 100
console.print(f"\n2. MODEL STABILITY")
console.print(f"   - RMSE Coefficient of Variation: {rmse_cv:.1f}%")
if rmse_cv < 20:
    console.print(f"   [green]✓ Excellent stability (CV < 20%)[/green]")
elif rmse_cv < 30:
    console.print(f"   [yellow]⚠ Moderate stability (CV 20-30%)[/yellow]")
else:
    console.print(f"   [red]✗ Poor stability (CV > 30%) - investigate further[/red]")

# Bias assessment
if abs(mean_error) < 1:
    console.print(f"\n3. MODEL BIAS: [green]✓ LOW BIAS[/green]")
else:
    console.print(
        f"\n3. MODEL BIAS: [yellow]⚠ MODERATE BIAS ({mean_error:+.2f} USDT)[/yellow]"
    )

# Directional accuracy assessment
if stats["Dir Acc"]["mean"] > 55:
    console.print(f"\n4. TRADING VIABILITY: [green]✓ PROMISING (>55% accuracy)[/green]")
elif stats["Dir Acc"]["mean"] > 50:
    console.print(
        f"\n4. TRADING VIABILITY: [yellow]⚠ MARGINAL (50-55% accuracy)[/yellow]"
    )
else:
    console.print(
        f"\n4. TRADING VIABILITY: [red]✗ NOT RECOMMENDED (<50% accuracy)[/red]"
    )

# Production recommendations
console.print(f"\n5. PRODUCTION RECOMMENDATIONS:")
if stats["Dir Acc"]["mean"] > 55 and rmse_cv < 30 and abs(mean_error) < 2:
    console.print(f"   [green]✓ Model is READY for production testing[/green]")
    console.print(f"   → Start with paper trading")
    console.print(f"   → Monitor live performance vs backtest")
    console.print(
        f"   → Set stop-loss at 2x average RMSE ({stats['RMSE']['mean'] * 2:.2f} USDT)"
    )
else:
    console.print(f"   [yellow]⚠ Model needs improvement before production[/yellow]")
    console.print(f"   → Tune hyperparameters")
    console.print(f"   → Try ensemble of top models")
    console.print(f"   → Increase training data / epochs")

console.print(f"\n6. NEXT STEPS:")
console.print(f"   → Compare with other models (see model_comparison.ipynb)")
console.print(f"   → Implement ensemble if multiple models perform well")
console.print(f"   → Set up monitoring dashboard for live predictions")

console.print("=" * 70)